# FinBERT scoring for Polygon market-proxy news


In [ ]:
!pip -q install transformers accelerate safetensors

import pandas as pd
import torch
from google.colab import files
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = 'ProsusAI/finbert'
INPUT_NAME = 'finbert_missing_articles_2020_2024_polygon_proxy_market.csv'
OUTPUT_NAME = 'finbert_missing_scores_2020_2024_polygon_proxy_market_colab.csv'
BATCH_SIZE = 32
MAX_LENGTH = 512

uploaded = files.upload()
assert INPUT_NAME in uploaded, f'Please upload {INPUT_NAME}'

articles = pd.read_csv(INPUT_NAME)
articles['article_id'] = articles['article_id'].astype(str)
articles['text'] = articles['text'].fillna('').astype(str)
articles = articles.drop_duplicates(subset=['article_id']).reset_index(drop=True)
print('articles to score:', len(articles))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
rows = []

for start in range(0, len(articles), BATCH_SIZE):
    batch = articles.iloc[start:start + BATCH_SIZE]
    encoded = tokenizer(
        batch['text'].tolist(),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=1).detach().cpu().numpy()
    for (_, row), prob in zip(batch.iterrows(), probs):
        label_probs = {id2label[i]: float(prob[i]) for i in range(len(prob))}
        positive = label_probs.get('positive', 0.0)
        negative = label_probs.get('negative', 0.0)
        neutral = label_probs.get('neutral', 0.0)
        rows.append({
            'article_id': row['article_id'],
            'published_utc': row.get('published_utc'),
            'text': row.get('text'),
            'finbert_positive': positive,
            'finbert_negative': negative,
            'finbert_neutral': neutral,
            'finbert_sentiment_score': positive - negative,
            'finbert_predicted_label': max(
                [('positive', positive), ('negative', negative), ('neutral', neutral)],
                key=lambda item: item[1],
            )[0],
        })
    if start % (BATCH_SIZE * 20) == 0:
        print(f'scored {min(start + BATCH_SIZE, len(articles))}/{len(articles)}')

scores = pd.DataFrame(rows)
scores.to_csv(OUTPUT_NAME, index=False)
print('wrote', OUTPUT_NAME, scores.shape)
files.download(OUTPUT_NAME)